# [기초-실습] 통계 101×데이터 분석: (4장) 추론통계~신뢰구간

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## ⚙️ 환경 준비 — 한글 폰트 설치 및 라이브러리 불러오기

In [ ]:
# 구글 코랩 환경에서 한글 폰트 설치 및 설정하기
# 필요시 아래 코드 실행 후, [런타임] - [세션 다시 시작] 후 셀을 다시 실행하세요.
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
# 파이썬 라이브러리 및 모듈 가져오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'NanumGothic'  # 기본 폰트 설정
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지

# 문제 1. 표본조사 체험하기

**📘 문제**

- 온라인 쇼핑몰은 전체 고객 수가 너무 많아, 모든 고객을 조사하기 어렵습니다.

- 그래서 무작위로 고객 30명을 뽑아 평균 만족도를 계산하고 이를 전체 만족도의 추정값으로 사용하려 합니다.

- 이번 실습에서는 직접 표본을 뽑고, 표본 평균을 구해보며,
  **“표본마다 결과가 달라질 수 있다”**는 추론 통계의 핵심 개념을 체험해봅니다.

In [ ]:
# 모집단 생성 (전체 고객 만족도 10,000명)
np.random.seed(2025)
population = np.random.normal(loc=7.5, scale=1.2, size=10000)
population = np.clip(population, 1, 10)  # 1점 ~ 10점 사이로 제한
df_pop = pd.DataFrame({'score': population})

# 전체 모집단 시각화
sns.histplot(df_pop['score'], bins=30, kde=True)
plt.title("전체 고객 만족도 분포 (모집단)")
plt.xlabel("만족도 점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 표본을 무작위로 여러 번 뽑아 보고, 표본 평균이 어떻게 변하는지 확인해봅시다.

- 히스토그램을 그리고, 표본 평균의 분포 형태를 관찰해봅시다.

In [ ]:
# [문제 1] Q1. 모집단에서 무작위로 30명을 뽑아 표본 평균을 구해봅시다.
# 여기에 코드를 작성해주세요.


# 모집단에서 무작위로 30명 추출 (비복원추출)
sample = np.random.choice(df_pop['score'], size=30, replace=False)

# 표본 평균 계산
sample_mean = sample.mean()

print("표본 평균:", sample_mean)
print("모집단 평균:", df_pop['score'].mean())

In [ ]:
# [문제 1] Q2. 이 과정을 500번 반복하고, 표본 평균을 리스트에 저장합니다.
# 여기에 코드를 작성해주세요.

# 표본추출(30명)을 500번 반복하여 표본 평균 저장
sample_means = []

for i in range(500):
    sample = np.random.choice(df_pop['score'], size=30, replace=False)
    sample_means.append(sample.mean())

sample_means = np.array(sample_means)

print("표본 평균 리스트 개수:", len(sample_means))
print("표본 평균들의 평균:", sample_means.mean())
print("표본 평균들의 표준편차:", sample_means.std())

In [ ]:
# [문제 1] Q3. 표본 평균들의 분포를 히스토그램으로 그려봅시다. 평균선도 함께 표시해 봅시다.
# 여기에 코드를 작성해주세요.

plt.figure(figsize=(8,5))

# 표본 평균들의 히스토그램 (KDE 곡선 포함)
sns.histplot(sample_means, bins=30, kde=True)

# 표본 평균들의 평균선 (빨간 점선)
plt.axvline(sample_means.mean(), color='red', linestyle='--', linewidth=2, 
            label=f'표본평균들의 평균: {sample_means.mean():.3f}')

# 모집단 평균선 (초록 실선) - 비교용
plt.axvline(df_pop['score'].mean(), color='green', linestyle='-', linewidth=2, 
            label=f'모집단 평균: {df_pop["score"].mean():.3f}')

plt.title("표본평균의 표집분포 (n=30, 500회 반복)")
plt.xlabel("표본 평균")
plt.ylabel("빈도")
plt.legend()
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 평균들은 어떤 값 주변에 많이 분포해 있나요? 이 값은 전체 모집단 평균과 얼마나 비슷한가요?

- 표본을 1번 뽑았을 때와 500번을 반복해서 뽑았을 때, 표본 평균의 분포나 신뢰성에는 어떤 차이가 있나요

- 친구가 다른 표본을 뽑았다면 같은 평균이 나왔을까요? 비슷한 결과가 나왔더라도 완전히 같지 않았다면, 그 이유는 무엇일까요?

- 표본 평균들의 분포는 어떤 모양인가요? 종 모양의 정규분포처럼 보이나요? 그렇다면 왜 그렇게 되는 걸까요?

In [ ]:
# [문제 1] 데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.

# - 표본 평균들은 모집단 평균(7.48061)에 몰려있습니다. 
# (500번 추출한 표본 평균들의 평균이) 7.488
# 약 0.007 차이, 거의 동일하다고 볼 수 있다.

# 1번 뽑았을 때는 분포를 확인할 수 없고 신뢰성이 떨어집니다.
# 어떤 모양인지, 평균과 표준편차는 어느정도인지 확인할 수 없기 때문입니다.
# 그러나 500번 뽑았을 때는 분포 확인이 가능하고
# 평균, 표준편차 확인이 가능하기에 신뢰성이 있습니다.

# 같지는 않고 조금 다르지 않았을까 생각합니다.
# 운이 좋게 만족도가 높은 사람만을 추출할 수도 있고
# 추출이 랜덤하게 이루어지기 때문에 완전히 같을 수는 없습니다.

# 종 모양의 정규분포로 보입니다.
# 모집단이 어떤 분포이든 간에, 
# 표본크기 n이 커질수록 표본평균의 분포는 
# 정규분포로 근사할 수 있기 때문입니다(중심극한정리).

# 문제 2. 중심극한정리

**📘 문제**

- 현실에서는 모집단의 분포가 정규분포가 아닐 수도 있습니다.

- 예를 들어, 일부 고객은 매우 높은 점수를 주고, 대부분은 낮은 점수를 주는 만족도 분포가 있을 수 있죠. (예: 지수분포)

- 이처럼 원래 분포가 비정규분포여도,
  표본을 여러 번 뽑아 평균을 계산하면, 그 평균들의 분포는 정규분포에 가까워진다는 것을
  **중심극한정리(Central Limit Theorem)**라고 합니다.

- 이번 실습에서는 다양한 크기의 표본을 뽑아 평균을 계산하고,
  그 평균들의 분포가 어떻게 변하는지를 직접 실험해 봅니다.

In [ ]:
# 지수분포를 따르는 모집단 생성
np.random.seed(2025)
population = np.random.exponential(scale=50, size=100000)  # 평균 50, 비대칭 분포

# 모집단 시각화
sns.histplot(population, bins=50, kde=True)
plt.title("고객 구매 금액 분포 (모집단: 지수분포)")
plt.xlabel("구매 금액")
plt.show()

**📌 아래를 수행해 보세요:**

- 비대칭적인 모집단(지수분포)에서 무작위로 표본을 추출해 평균을 구해봅시다.

- 표본 크기를 바꿔가며, 표본 평균들의 분포가 어떻게 변화하는지 확인해봅시다.

- 히스토그램을 그리고, 분포의 모양을 관찰해봅시다.

- 표본 크기가 커질수록 표본 평균 분포의 모양과 **퍼진 정도(분산)**가 어떻게 변하는지 관찰해봅시다.

In [ ]:
# [문제 2] Q1. 모집단에서 표본을 1000번 뽑고, 각 표본의 평균을 구해봅시다.
# 표본 크기 = 5일 때

sample_means_n5 = []

for i in range(1000):
    sample = np.random.choice(population, size=5, replace=False)
    sample_means_n5.append(sample.mean())

sample_means_n5 = np.array(sample_means_n5)

print("모집단 평균:", population.mean())
print("표본 평균들의 평균 (n=5):", sample_means_n5.mean())
print("표본 평균들의 표준편차 (n=5):", sample_means_n5.std())

# 여기에 코드를 작성해주세요.

In [ ]:
# [문제 2] Q2. 위 과정을 표본 크기 30, 100일 때도 반복해봅시다.
# sample_size = 30, 100

# 여기에 코드를 작성해주세요.

sample_means_n30 = []

for i in range(1000):
    sample = np.random.choice(population, size=30, replace=False)
    sample_means_n30.append(sample.mean())

sample_means_n30 = np.array(sample_means_n30)

print("모집단 평균:", population.mean())
print("표본 평균들의 평균 (n=30):", sample_means_n30.mean())
print("표본 평균들의 표준편차 (n=30):", sample_means_n30.std())


sample_means_n100 = []

for i in range(1000):
    sample = np.random.choice(population, size=100, replace=False)
    sample_means_n100.append(sample.mean())

sample_means_n100 = np.array(sample_means_n100)

print("모집단 평균:", population.mean())
print("표본 평균들의 평균 (n=100):", sample_means_n100.mean())
print("표본 평균들의 표준편차 (n=100):", sample_means_n100.std())



In [ ]:
# [문제 2] Q3. 각 표본 크기별로 표본 평균들의 분포를 히스토그램으로 그려봅시다.
# 평균선을 함께 표시해 봅시다.

# 여기에 코드를 작성해주세요.

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sample_data = {
    'n=5': sample_means_n5,
    'n=30': sample_means_n30,
    'n=100': sample_means_n100
}

for ax, (label, data) in zip(axes, sample_data.items()):
    sns.histplot(data, bins=30, kde=True, ax=ax)
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=2,
               label=f'표본평균들의 평균: {data.mean():.2f}')
    ax.axvline(population.mean(), color='green', linestyle='-', linewidth=2,
               label=f'모집단 평균: {population.mean():.2f}')
    ax.set_title(f"표본평균의 분포 ({label})")
    ax.set_xlabel("표본 평균")
    ax.set_ylabel("빈도")
    ax.legend()

plt.tight_layout()
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 크기가 작을 때 (예: 5), 평균들의 분포는 어떤 모양인가요?

- 표본 크기가 커질수록 평균 분포의 모양은 어떤 변화를 보이나요?

- 원래 모집단은 비대칭이었는데, 왜 평균들의 분포는 정규분포처럼 바뀌었을까요?

- 이 실험을 통해 중심극한정리를 어떻게 이해하게 되었나요?

- 표본 크기에 따라 **분포의 넓이(흩어짐)**는 어떻게 달라지나요?

In [ ]:
# [문제 2]데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.

# 분포가 넓게 퍼져 있고, 
# 오른쪽으로 꼬리가 긴 비대칭 모양입니다. 
# 원래 지수분포의 특징(비대칭성)이 표본 평균에도 그대로 남아있습니다. 
# 큰 값이 드물게 뽑히면 평균이 크게 튀는 경우가 자주 있습니다.

# n=5 → 30 → 100으로 갈수록 
# 분포가 점점 종 모양(정규분포)에 가까워집니다.

# 표본 크기가 커지면 평균값은 원래 분포의 비대칭성보다 
# 중심 경향을 더 잘 반영하게 됩니다.

# 표본 크기가 충분히 크면 표본 평균의 분포는 
# 정규분포에 가까워진다를 이해하게 되었습니다.

# 분포의 넓이(흩어짐)가 좁아집니다.
# 

# 문제 3. 표준오차

**📘 문제**

- 앞선 실습에서 우리는 **표본 크기(n)가 커질수록 표본 평균들의 분포가 더 좁아진다**는 것을 확인했습니다.
- 이처럼 표본 평균들이 얼마나 흩어져 있는지(분포의 퍼진 정도)를 나타내는 값을 **표준오차(Standard Error, SE)**라고 부릅니다.
- 표준오차는 **표본 평균들의 표준편차**와 같은 의미이며, 이는 우리가 뽑은 표본 평균이 실제 모평균과 평균적으로 얼마나 떨어져 있을지를 나타내는 **'예상 오차의 크기'**입니다.

- 통계학적으로 이 표준오차는 **`SE = σ / √n`** (모집단 표준편차 / 표본 크기의 제곱근) 이라는 공식으로 계산할 수 있습니다.
- 이 공식은 **표본 크기(n)가 커질수록 표준오차(SE)가 작아진다**는 것을 명확히 보여줍니다.

- 이번 실습에서는 여러 크기의 표본을 뽑아, 시뮬레이션을 통해 얻은 **표본 평균들의 표준편차(실험값)**가 공식으로 계산한 **표준오차(이론값)**와 얼마나 일치하는지 직접 확인해봅니다.

In [ ]:
# 모집단 생성 (평균 100, 표준편차 15)
np.random.seed(2025)
population = np.random.normal(loc=100, scale=15, size=100000)

# 모집단 시각화
sns.histplot(population, bins=40, kde=True)
plt.title("모집단 분포 (평균 100, 표준편차 15)")
plt.xlabel("값")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단에서 여러 크기의 표본(10, 30, 100, 500)을 각각 1000번 뽑고, 그 평균들을 구한 뒤, **표본 평균들의 표준편차(=실험적 표준오차)**를 계산해봅시다.

- 이 결과를 이론적인 표준오차 공식과 비교하는 표를 만들고, 시각화해봅시다.

In [ ]:
# [문제 3] Q1. 표본 크기 10, 30, 100, 500에 대해 각각 1000번 표본을 뽑고, 평균을 구해봅시다.
# 각 표본 평균 분포의 표준편차를 계산해봅시다.
# 결과를 리스트에 저장하고, 표로 정리해봅시다.

import numpy as np
import pandas as pd

# 표본 크기 리스트
sample_sizes = [10, 30, 100, 500]
n_repeat = 1000

# 결과를 저장할 리스트
results = []

for n in sample_sizes:
    means = []
    for i in range(n_repeat):
        sample = np.random.choice(population, size=n, replace=False)
        means.append(sample.mean())
    means = np.array(means)
    
    std_of_means = means.std()  # 표본 평균 분포의 표준편차
    
    results.append({
        '표본크기': n,
        '표본평균들의 표준편차': std_of_means
    })

# 표로 정리
df_result = pd.DataFrame(results)
print(df_result)
# 여기에 코드를 작성해주세요.

In [ ]:
# [문제 3] Q2. 이론적인 표준오차와 비교해봅시다.
# [공식] 표준오차(SE) = 모집단 표준편차 / √표본크기

# 여기에 코드를 작성해주세요.

sample_sizes = [10, 30, 100, 500]
n_repeat = 1000

results = []

for n in sample_sizes:
    means = []
    for i in range(n_repeat):
        sample = np.random.choice(population, size=n, replace=False)
        means.append(sample.mean())
    means = np.array(means)
    
    empirical_se = means.std()
    theoretical_se = population.std() / np.sqrt(n)   # 이론적 표준오차 = 모집단 표준편차 / √n
    
    results.append({
        '표본크기': n,
        '실험적 표준오차': empirical_se,
        '이론적 표준오차': theoretical_se,
        '차이': abs(empirical_se - theoretical_se)
    })

df_result = pd.DataFrame(results)
print(df_result)



In [ ]:
# [문제 3] Q3. 실험값과 이론값을 시각화해봅시다.
# 표본 크기를 x축, 표준오차를 y축으로 한 꺾은선 그래프를 그려봅시다.

# 여기에 코드를 작성해주세요.

import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.plot(df_result['표본크기'], df_result['실험적 표준오차'], marker='o', label='실험적 표준오차', linewidth=2)
plt.plot(df_result['표본크기'], df_result['이론적 표준오차'], marker='s', linestyle='--', label='이론적 표준오차', linewidth=2)
plt.title("표본 크기에 따른 표준오차 변화")
plt.xlabel("표본 크기 (n)")
plt.ylabel("표준오차")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**🧠 데이터를 어떻게 읽을까요?**

- 표본 크기가 작을수록, 표본 평균의 분포는 어떤 모양인가요? 넓게 퍼져 있나요?

- 표본 크기가 커질수록, 평균 분포는 어떻게 변하나요?

- 실험값과 이론값(공식 계산값)은 얼마나 비슷한가요?

- 왜 표본 크기가 커질수록 표준오차는 작아질까요?

In [ ]:
# [문제 3] 데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.

# 표본 크기가 작을수록
# 표본 평균의 분포가 넓게 퍼져 있습니다.

# 분포의 모양이 좁아지고 종모양(정규분포)이 됩니다.

# 표본이 작았을 때는 0.2 정도로 차이가 났지만
# 표본이 커지면서 차이가 거의 안 나고
# 그래프도 대부분 겹치는 경향을 확인.

# 표준오차 공식(SE = σ/√n)에 따르면 표본 크기가 커질수록 
# 분모인 √n이 커지므로 SE는 자연히 작아집니다.


# 문제 4. 신뢰구간 계산과 해석

**📘 문제**

- 표본 평균은 모집단 평균을 추정하는 좋은 점 추정(Point Estimation) 값이지만, 표본오차 때문에 정확히 일치하지는 않습니다.

- 그래서 우리는 "모집단 평균이 아마 이 범위 안에 있을 것이다"라고 **구간으로 추정(Interval Estimation)**하는 것이 더 합리적입니다. 이때 사용하는 개념이 바로 **신뢰구간(Confidence Interval)**입니다.

- 신뢰구간은 표본평균 ± 오차범위 형태로 계산되며, 이 오차범위는 신뢰수준(예: 95%, 99%)과 표본오차에 의해 결정됩니다.

- 이번 실습에서는 **모집단 표준편차(σ)를 알 때(z-분포)**와 **모를 때(t-분포)**의 신뢰구간을 각각 계산해보고, 신뢰수준에 따라 구간의 폭이 어떻게 변하는지 확인해봅니다.

In [ ]:
# 모집단 생성
np.random.seed(2025)
population = np.random.normal(loc=70, scale=10, size=10000)

# 모집단 시각화
sns.histplot(population, bins=40, kde=True)
plt.title("모집단 분포 (평균 70, 표준편차 10)")
plt.xlabel("점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단에서 30명을 무작위로 뽑아 평균, 표준편차, 표준오차를 계산해보세요.

- 95% 신뢰구간을 z-분포와 t-분포를 각각 사용해서 계산해보세요.

- 신뢰수준을 바꿨을 때(90%, 99%) 신뢰구간이 어떻게 변하는지 확인해보세요.

In [ ]:
# [문제 4] Q1. 모집단에서 표본 30명을 무작위로 추출하고, 표본 평균과 표준편차를 구해봅시다.
# 표준오차도 함께 계산해보세요.

# 여기에 코드를 작성해주세요.

# 표본 30명 무작위 추출
sample = np.random.choice(population, size=30, replace=False)

# 표본 평균, 표본 표준편차 (ddof=1: 표본 표준편차는 n-1로 나눔)
sample_mean = sample.mean()
sample_std = sample.std(ddof=1)

# 표준오차 = 표본 표준편차 / √표본크기
se = sample_std / np.sqrt(len(sample))

print("표본 평균:", sample_mean)
print("표본 표준편차:", sample_std)
print("표준오차(SE):", se)

In [ ]:
# [문제 4] Q2. 모집단의 표준편차를 알고 있다고 가정하고, z-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

# 여기에 코드를 작성해주세요.
from scipy import stats

# 모집단 표준편차를 알고 있다고 가정
pop_std = 10
n = len(sample)

# z-분포 사용, 95% 신뢰수준 -> z = 1.96
confidence = 0.95
z_value = stats.norm.ppf(1 - (1 - confidence) / 2)

# 표준오차 (모집단 표준편차 기준)
se = pop_std / np.sqrt(n)

# 95% 신뢰구간 계산
margin_of_error = z_value * se
ci_lower = sample_mean - margin_of_error
ci_upper = sample_mean + margin_of_error

print("z-value (95%):", z_value)
print("표준오차(SE):", se)
print("오차범위:", margin_of_error)
print(f"95% 신뢰구간: ({ci_lower:.3f}, {ci_upper:.3f})")



In [ ]:
# [문제 4] Q3. 모집단의 표준편차를 모른다고 가정하고, 표본 표준편차와 t-분포를 사용하여 95% 신뢰구간을 계산해봅시다.

# 여기에 코드를 작성해주세요.

from scipy import stats

sample_std = sample.std(ddof=1)   # 표본 표준편차 (n-1로 나눔)
n = len(sample)

# t-분포 사용, 95% 신뢰수준, 자유도 = n-1
confidence = 0.95
df = n - 1
t_value = stats.t.ppf(1 - (1 - confidence) / 2, df)

# 표준오차 (표본 표준편차 기준)
se = sample_std / np.sqrt(n)

# 95% 신뢰구간 계산
margin_of_error = t_value * se
ci_lower = sample_mean - margin_of_error
ci_upper = sample_mean + margin_of_error

print("자유도(df):", df)
print("t-value (95%):", t_value)
print("표준오차(SE):", se)
print("오차범위:", margin_of_error)
print(f"95% 신뢰구간: ({ci_lower:.3f}, {ci_upper:.3f})")

In [ ]:
# [문제 4] Q4. 신뢰수준을 90%, 99%로 바꿔가며 신뢰구간을 계산해보고, 그 폭을 비교해봅시다.

# 여기에 코드를 작성해주세요.
confidence_levels = [0.90, 0.95, 0.99]
results = []

for conf in confidence_levels:
    t_value = stats.t.ppf(1 - (1 - conf) / 2, df)
    margin_of_error = t_value * se
    ci_lower = sample_mean - margin_of_error
    ci_upper = sample_mean + margin_of_error
    width = ci_upper - ci_lower
    
    results.append({
        '신뢰수준': f"{int(conf*100)}%",
        't-value': t_value,
        '오차범위': margin_of_error,
        '신뢰구간 하한': ci_lower,
        '신뢰구간 상한': ci_upper,
        '신뢰구간 폭': width
    })

df_result = pd.DataFrame(results)
print(df_result)


**🧠 데이터를 어떻게 읽을까요?**

- z-분포와 t-분포를 사용한 신뢰구간은 얼마나 차이가 있나요?

- 신뢰수준이 높아질수록 신뢰구간의 폭은 어떻게 변하나요? 왜 그럴까요?

- 신뢰구간이 넓다는 건 좋은 걸까요? 나쁜 걸까요?

- 이 데이터가 실제 고객 만족도라면, 신뢰구간 정보를 마케팅 전략에 어떻게 활용할 수 있을까요?

In [ ]:
# [문제 4] 데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.

# z-분포(64.03~71.18, 폭 3.58)보다 
# t-분포(63.47~71.74, 폭 4.14)의 신뢰구간이 
# 더 넓습니다.

# 신뢰수준이 높아질수록 신뢰구간의 폭도 넓어집니다.
# 신뢰수준이란 "이 방법으로 신뢰구간을 반복해서 구했을 때, 그 중 몇 %가 실제 모평균을 포함하는가"를 의미합니다.
#더 높은 비율(99%)로 모평균을 포함시키려면, 표본 평균에서 더 멀리까지 범위를 넓혀 잡아야 그만큼 놓칠 위험이 줄어듭니다. 
# 반대로 신뢰수준을 낮추면(90%) 더 좁은 구간으로도 충분하다고 보는 대신, 그 구간이 모평균을 놓칠 확률(10%)이 상대적으로 커집니다.

# 단순히 좋다 나쁘다로 판별하기 어렵습니다.
# 넓으면 모평균을 포함할 확률이 높아 검증의 확신이 올라감. 
# 범위가 넓어져 "모평균은 62~73점 사이"라는 정보는 의사결정에 도움이 되지 않을 수 있습니다.

# "95% 확신으로 평균 만족도는 63.5~71.7점 사이"
# 신뢰구간 하한이 회사 목표치(예: 70점)보다 낮다면,
# 현재 표본만으로는 목표 달성을 확신할 수 없다는 신호이므로 
# 추가 조사나 마케팅 활동이 필요하다고 판단할 수 있습니다.

# 문제 5. 미니 프로젝트 - 고객 만족도 신뢰구간 추정

**📘 문제**

- 전체 고객 10,000명을 대상으로 만족도 조사를 하는 것은 시간과 비용이 많이 듭니다.
- 그래서 우리는 무작위로 일부 고객만 조사하여, 전체 고객의 평균 만족도를 추정하려 합니다.

- 이 프로젝트에서는 실제와 같은 상황을 가정하여, 표본을 뽑고 신뢰구간을 계산한 뒤, 이 결과를 바탕으로 마케팅 전략에 어떻게 활용할 수 있을지까지 생각해보는 실습을 진행합니다.

In [ ]:
# 모집단 생성 (고객 만족도 10,000명)
np.random.seed(2025)
population = np.random.normal(loc=7.2, scale=1.0, size=10000)
population = np.clip(population, 1, 10)

# 모집단 시각화
sns.histplot(population, bins=30, kde=True)
plt.title("전체 고객 만족도 분포 (모집단)")
plt.xlabel("만족도 점수")
plt.show()

**📌 아래를 수행해 보세요:**

- 모집단을 생성하고, 거기서 표본을 40명 뽑아 평균을 계산해봅시다.

- 표본 평균과 표준편차를 바탕으로 95% 신뢰구간을 계산해봅시다.

- 히스토그램을 그리고 신뢰구간을 시각화해봅시다.

- 이 결과를 어떻게 해석하고, 마케팅 전략에 활용할 수 있을지 생각해봅시다.

In [ ]:
# [문제 5] Q1. 모집단에서 표본 40명을 무작위로 추출하고, 표본 평균과 표준편차를 구해봅시다.
# 여기에 코드를 작성해주세요.

# 표본 40명 무작위 추출
sample = np.random.choice(population, size=40, replace=False)

# 표본 평균, 표본 표준편차 (ddof=1: 표본 표준편차는 n-1로 나눔)
sample_mean = sample.mean()
sample_std = sample.std(ddof=1)

print("표본 평균:", sample_mean)
print("표본 표준편차:", sample_std)

In [ ]:
# [문제 5] Q2. 표준오차(SE)를 구하고, t-분포를 사용하여 95% 신뢰구간을 계산해봅시다.
# 여기에 코드를 작성해주세요.

from scipy import stats

# 표준오차 계산
se = sample_std / np.sqrt(n)

# t-분포 사용, 95% 신뢰수준, 자유도 = n-1
confidence = 0.95
df = n - 1
t_value = stats.t.ppf(1 - (1 - confidence) / 2, df)

# 95% 신뢰구간 계산
margin_of_error = t_value * se
ci_lower = sample_mean - margin_of_error
ci_upper = sample_mean + margin_of_error

print("표준오차(SE):", se)
print("자유도(df):", df)
print("t-value (95%):", t_value)
print("오차범위:", margin_of_error)
print(f"95% 신뢰구간: ({ci_lower:.3f}, {ci_upper:.3f})")

In [ ]:
# [문제 5] Q3. 표본 데이터의 히스토그램을 그리고, 평균 및 신뢰구간을 함께 시각화해봅시다.
#  여기에 코드를 작성해주세요.

plt.figure(figsize=(8,5))
sns.histplot(sample, bins=15, kde=True, color='skyblue')

# 표본 평균선
plt.axvline(sample_mean, color='red', linestyle='-', linewidth=2, 
            label=f'표본 평균: {sample_mean:.3f}')

# 신뢰구간 표시 (음영 영역 + 경계선)
plt.axvspan(ci_lower, ci_upper, color='orange', alpha=0.2, 
            label=f'95% 신뢰구간: ({ci_lower:.3f}, {ci_upper:.3f})')
plt.axvline(ci_lower, color='orange', linestyle='--', linewidth=1.5)
plt.axvline(ci_upper, color='orange', linestyle='--', linewidth=1.5)

plt.title("표본 분포(n=40)와 평균 및 95% 신뢰구간")
plt.xlabel("만족도 점수")
plt.ylabel("빈도")
plt.legend()
plt.show()

In [ ]:
# [문제 5] Q4. 신뢰구간의 결과에 따라 어떤 구체적인 마케팅 전략을 세울 수 있을까요?
# 여기에 의견을 작성해주세요.

# 목표 점수 대비 현재 위치 판단
# 경쟁사의 평균 만족도가 예를 들어 6.5점으로 알려져 있다면, 
# 우리 신뢰구간 하한(6.793)이 이미 그보다 높으므로 
# "우리는 경쟁사보다 만족도가 통계적으로 유의미하게 높다"는 근거를 가지고 
# 마케팅 메시지("고객이 인정한 만족도 1위" 등)에 활용할 수 있습니다.

**🧠 데이터를 어떻게 읽을까요?**

- 신뢰구간은 몇 점에서 몇 점 사이인가요?

- 이 구간은 전체 모집단 평균을 포함하고 있나요?

- 이 결과를 바탕으로 고객 만족도가 충분히 높다고 말할 수 있을까요?

- 만약 신뢰구간이 너무 넓게 나왔다면, 그 이유는 무엇이고 어떻게 개선할 수 있을까요?

In [ ]:
# [문제 5] 데이터를 어떻게 읽을까요?
# 여기에 의견을 작성해주세요.

# (6.734, 7.555)

#모집단 평균 7.2, 포함하고 있습니다. 

# 기준에 따라 다름.
# 기준이 8점이라면 높다고 말하기 애매
# 기준이 6점이라면 높다고 말할 수 있음

# 표본 크기가 작거나 표준편차가 크거나
# 개선방법: 표본 크기를 늘리기 